# Análisis Exploratorio de Datos (EDA) - Airbnb Mexico


## Objetivo
Entender la estructura, calidad y características de los datos antes de realizar transformaciones.

**Colecciones a analizar:**
- `Listings`: Información detallada de los apartamentos
- `Reviews`: Información de reseñas de clientes
- `Calendar`: Información de disponibilidad

## Importaciones

In [ ]:
import sys
from pathlib import Path

# Raíz del proyecto: sirve si el cwd es `notebooks/` o la raíz del repo
_cwd = Path.cwd().resolve()
if (_cwd / "src").is_dir():
    _root = _cwd
elif (_cwd.parent / "src").is_dir():
    _root = _cwd.parent
else:
    _root = _cwd.parent
sys.path.insert(0, str(_root))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import Extraccion, get_logger

# Configuración de visualizaciones
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 5)

logger = get_logger(__name__)
print("✓ Importaciones completadas")

## Funciones comunes

In [ ]:
from IPython.display import display

def analizar_estructura(df, nombre_coleccion):
    """
    Realiza análisis general de estructura de un DataFrame.
    
    Muestra:
    - Primeras filas
    - Dimensiones
    - Tipos de datos
    - Estadísticas numéricas
    - Estadísticas generales
    - Resumen de variables no numéricas
    """
    print("\n" + "=" * 80)
    print(f"COLECCIÓN: {nombre_coleccion}")
    print("=" * 80)
    
    print(f"\nPRIMERAS FILAS (primeros 5 registros):")
    display(df.head(5))
    
    print(f"\nDIMENSIONES:")
    print(f"   - Filas: {df.shape[0]:,}")
    print(f"   - Columnas: {df.shape[1]}")
    
    print(f"\nTIPOS DE DATOS:")
    print("INFO")
    df.info(verbose=False)
    print("TYPES")
    print(df.dtypes)
    
    print(f"\nESTADÍSTICAS (Columnas numéricas):")
    display(df.describe().T)
    
    print(f"\nESTADÍSTICAS COMPLETAS (incluye variables categóricas):")
    display(df.describe(include='all').T)
    
    cols_no_numericas = df.select_dtypes(include=['object', 'string', 'bool', 'datetime64', 'str']).columns
    if len(cols_no_numericas) > 0:
        print(f"\nCOLUMNAS NO NUMÉRICAS ({len(cols_no_numericas)}):")
        for col in cols_no_numericas:
            try:
                unicos = df[col].nunique(dropna=False)
                print(f"   - {col}: {unicos} valores únicos")
                top = df[col].value_counts(dropna=False).head(5)
                print(f"      Principales valores:\n{top.to_dict()}")
            except Exception:
                print(f"   - {col}: Contiene datos complejos o no se puede resumir")


def analizar_valores_nulos(df, nombre_coleccion):
    """
    Analiza valores nulos y faltantes en un DataFrame.
    
    Muestra:
    - Cantidad de nulos por columna
    - Porcentaje respecto al total de registros
    - Solo columnas que tienen al menos un nulo
    """
    print(f"\nVALORES NULOS Y FALTANTES - {nombre_coleccion}:")
    print("-" * 80)
    
    nulos = df.isnull().sum()
    total_registros = len(df)
    nulos_pct = (nulos / total_registros * 100).round(2)
    
    df_nulos = pd.DataFrame({
        'Columna': nulos.index,
        'Cantidad de nulos': nulos.values,
        'Porcentaje (%)': nulos_pct.values
    })
    
    # Filtrar solo columnas con nulos
    df_nulos_filtrado = df_nulos[df_nulos['Cantidad de nulos'] > 0].sort_values(
        'Cantidad de nulos', ascending=False
    )
    
    if len(df_nulos_filtrado) > 0:
        print(f"   Columnas con valores faltantes:")
        display(df_nulos_filtrado)
    else:
        print(f"   Todas las columnas están completas (sin nulos)")


def analizar_duplicados(df, nombre_coleccion, col_id='_id'):
    """
    Analiza registros duplicados de dos formas:
    1. Duplicados por columna ID (identidad única)
    2. Filas completamente duplicadas en todas sus columnas
    """
    print(f"\nREGISTROS DUPLICADOS - {nombre_coleccion}:")
    print("-" * 80)
    
    total = len(df)
    
    # Análisis 1: Duplicados por ID
    unicos = df[col_id].nunique()
    duplicados_id = total - unicos
    pct_duplicados_id = (duplicados_id / total * 100) if total > 0 else 0
    
    print(f"   1. POR COLUMNA ID ({col_id}):")
    print(f"      - Total de registros: {total:,}")
    print(f"      - Registros únicos: {unicos:,}")
    print(f"      - Duplicados por ID: {duplicados_id:,} ({pct_duplicados_id:.2f}%)")
    
    # Análisis 2: Filas completamente duplicadas
    filas_duplicadas = df.duplicated().sum()
    pct_filas_dup = (filas_duplicadas / total * 100) if total > 0 else 0
    
    print(f"\n   2. FILAS COMPLETAMENTE DUPLICADAS (todas las columnas):")
    print(f"      - Filas duplicadas: {filas_duplicadas:,} ({pct_filas_dup:.2f}%)")
    
    if duplicados_id == 0 and filas_duplicadas == 0:
        print(f"\n   Excelente: No hay duplicados en esta colección")


def analizar_outliers(df, nombre_coleccion, columnas=None):
    """
    Analiza outliers en columnas numéricas usando método de cuartiles (IQR).
    """
    if columnas is None:
        columnas = df.select_dtypes(include='number').columns.tolist()
    
    print(f"\nANÁLISIS DE OUTLIERS - {nombre_coleccion}:")
    print("-" * 80)
    
    if len(columnas) == 0:
        print("   No hay columnas numéricas para analizar")
        return
    
    for col in columnas:
        if col not in df.columns:
            print(f"   Columna '{col}' no encontrada")
            continue
            
        datos = pd.to_numeric(df[col], errors='coerce').dropna()
        
        if len(datos) == 0:
            print(f"\n   {col}: No hay datos numéricos válidos")
            continue
        
        Q1 = datos.quantile(0.25)
        Q3 = datos.quantile(0.75)
        IQR = Q3 - Q1
        
        if IQR == 0:
            print(f"\n   {col}: IQR = 0, no hay variación suficiente para detectar outliers con este método")
            continue
        
        limite_inferior = Q1 - 1.5 * IQR
        limite_superior = Q3 + 1.5 * IQR
        
        outliers = datos[(datos < limite_inferior) | (datos > limite_superior)]
        pct_outliers = len(outliers) / len(datos) * 100
        
        print(f"\n   {col}:")
        print(f"      - Mínimo: {datos.min():.2f}")
        print(f"      - Q1 (25%): {Q1:.2f}")
        print(f"      - Mediana: {datos.median():.2f}")
        print(f"      - Q3 (75%): {Q3:.2f}")
        print(f"      - Máximo: {datos.max():.2f}")
        print(f"      - IQR: {IQR:.2f}")
        print(f"      - Outliers detectados: {len(outliers):,} ({pct_outliers:.2f}%)")
        print(f"      - Rango válido: [{limite_inferior:.2f}, {limite_superior:.2f}]")


## Extracción de datos desde MongoDB

In [ ]:
# Conectar a MongoDB y extraer datos
extraccion = Extraccion()
print("Conectando a MongoDB...")
extraccion.conectar()

print("\nExtrayendo colecciones...")
df_listings = extraccion.extraer_coleccion("Listings")
df_reviews = extraccion.extraer_coleccion("Reviews")
df_calendar = extraccion.extraer_coleccion("Calendar")

print("\nExtracción completada")
print(f"\nRESUMEN:")
print(f"   - Listings: {len(df_listings):,} registros, {len(df_listings.columns)} columnas")
print(f"   - Reviews: {len(df_reviews):,} registros, {len(df_reviews.columns)} columnas")
print(f"   - Calendar: {len(df_calendar):,} registros, {len(df_calendar.columns)} columnas")

## 1. Análisis de Colección: LISTINGS

Información detallada sobre los apartamentos disponibles en Airbnb Buenos Aires.

### Análisis de estructura

In [ ]:
analizar_estructura(df_listings, "LISTINGS")

### Interpretación del resultado

- La colección Listings (Anuncios) corresponde a los anuncios que la gente publica en las cuales promociona sus propiedad, habitaciones, experiencias, etc.
- La colección tiene la siguiente estructura: 
    - Filas: 27.051
    - Columnas: 17

*Explicación de sus campos*
- **id**  
  Identificador único del listing en Airbnb.

- **name**  
  Nombre del alojamiento.

- **host_id**  
  Identificador único del anfitrión.

- **host_name**  
  Nombre del anfitrión.

- **neighbourhood**  
  Barrio o zona donde se encuentra el alojamiento.

- **latitude**  
  Latitud geográfica.

- **longitude**  
  Longitud geográfica.

- **room_type**  
  Tipo de propiedad:
  - `Entire home/apt`: alojamiento completo
  - `Private room`: habitación privada
  - `Shared room`: habitación compartida

- **minimum_nights**  
  Número mínimo de noches requeridas por reserva.

- **price**  
  Precio por noche (en MXN).

- **number_of_reviews**  
  Número total de reseñas recibidas.

- **last_review**  
  Fecha de la última reseña.

- **reviews_per_month**  
  Promedio de reseñas por mes.

- **number_of_reviews_ltm**  
  Número de reseñas en los últimos 12 meses.

- **calculated_host_listings_count**  
  Número de propiedades que tiene el anfitrión.

- **availability_365**  
  Número de días en el año en los que el alojamiento está disponible.

### Interpretación general

Este dataset permite analizar:
- Oferta de alojamientos
- Precios por zona
- Actividad y demanda (reseñas)
- Tipo de anfitriones (casuales vs profesionales)
- Disponibilidad anual




### Análisis de valores nulos

In [ ]:
analizar_valores_nulos(df_listings, "LISTINGS")



### Interpretación del resultado

- Las filas cuyo price sea nulo, deben de ser descartadas ya que el precio no es un valor que se pueda inferir o calcular con base en otros datos. Es un valor que el anfitrión asigna a su propiedad con base en su criterio
- Dado que last_review y reviews_per_month tienen el mismo número de nulos, se realiza un análisis para comprobar que si uno de los valores es nulo el otro también. Siendo esto algo real.

Para este caso lo que haremos será obtener estos valores desde la colección de Reviews:
- Relacionando vía Listing_id, obtendremos:
- La fecha de la ultima Review
- El promedio de Reviews por mes

En caso de que luego de esto, hayan filas en donde el valor de last_review o de reviews_per_month sea nulo. Estás filas serán eliminadas

- Para los host_names sin valor, se asignará el valor de "empty_hostname" para agruparlos de mejor manera

### Análisis de duplicados

In [ ]:
analizar_duplicados(df_listings, "LISTINGS", col_id='id')

### Análisis de outliers

In [ ]:
analizar_outliers(df_listings, "LISTINGS", columnas=["price", "minimum_nights", "availability_365"])

### Interpretación del resultado

El resultado del análisis estádistico y de outliers, permite o nos da la opción de crear categorias para poder clasificar los listings.

### 1. price → Categoría de precio

**Estadísticos:**
- Q1: 643
- Mediana: 1039
- Q3: 1611
- Límite superior (sin outliers): 3063

**Nueva variable: `price_category`**

- Budget         → price ≤ 643
- Mid-range      → 643 < price ≤ 1039
- Upper-mid      → 1039 < price ≤ 1611
- Premium        → 1611 < price ≤ 3063
- Luxury/Outlier → price > 3063

---

### 2. minimum_nights → Tipo de estancia

**Estadísticos:**
- Q1: 1
- Mediana: 2
- Q3: 2
- Límite superior (sin outliers): 3.5

**Nueva variable: `stay_category`**

- Short stay     → minimum_nights ≤ 1
- Standard stay  → 1 < minimum_nights ≤ 2
- Extended stay  → 2 < minimum_nights ≤ 3
- Long stay/outlier → minimum_nights > 3

---

### 3. availability_365 → Nivel de disponibilidad

**Estadísticos:**
- Q1: 140
- Mediana: 269
- Q3: 341
- Sin outliers detectados

**Nueva variable: `availability_category`**

- Low availability    → availability_365 ≤ 140
- Medium availability → 140 < availability_365 ≤ 269
- High availability   → 269 < availability_365 ≤ 341
- Very high availability → availability_365 > 341

---

### Notas metodológicas

- Se utilizó el criterio de IQR (Interquartile Range) para detectar outliers.
- Las categorías se definieron con base en cuantiles para mantener balance en la distribución.
- Los valores fuera del rango superior se consideran "outliers" pero se mantienen como categoría separada para análisis.




## 2. Análisis de Colección: REVIEWS

Información sobre las reseñas que dejan los usuarios de los apartamentos.

### Análisis de estructura

In [ ]:
analizar_estructura(df_reviews, "REVIEWS")

### Interpretación del resultado

- La colección Reviews (Reseñas) corresponde a las opiniones de la gente acerca del servicio obtenido
- La colección tiene la siguiente estructura: 
    - Filas: 1.454.740
    - Columnas: 3

*Explicación de sus campos*
- **_id (ObjectId)**  
  Identificador único del documento en MongoDB.  
  No tiene valor analítico directo.

- **listing_id (Int32)**  
  Identificador del alojamiento al que pertenece la reseña.  
  Permite relacionar esta colección con `Listings` y `Calendar`.

- **date (Date)**  
  Fecha en la que se realizó la reseña.  
  Es clave para análisis de tendencia, estacionalidad y actividad en el tiempo.

**Consideraciones**

- No todas las reservas generan una reseña → los datos pueden subestimar la demanda real
- No incluye contenido textual de la reseña (solo metadata)
- Puede haber sesgo hacia usuarios más activos o experiencias extremas

### Análisis de valores nulos

In [ ]:
analizar_valores_nulos(df_reviews, "REVIEWS")

### Análisis de duplicados

In [ ]:
analizar_duplicados(df_reviews, "REVIEWS", col_id='_id')

### Análisis de outliers

In [ ]:
analizar_outliers(df_reviews, "REVIEWS")

## 3. Análisis de Colección: CALENDAR

Información sobre disponibilidad, precios y políticas de noches mínimas para cada día.

### Análisis de estructura

In [ ]:
analizar_estructura(df_calendar, "CALENDAR")

### Interpretación del resultado

- La colección Calendar (Calendario) corresponde a la disponibilidad diaria de un alojamiento. Cada fila es un día específico de una propiedad
- La colección tiene la siguiente estructura: 
    - Filas: 9,873,624
    - Columnas: 6

*Explicación de sus campos*
- **_id (ObjectId)**  
  Identificador único del documento en MongoDB.  
  No tiene valor analítico directo.

- **listing_id (Int32)**  
  Identificador del alojamiento.  
  Permite relacionar esta colección con `Listings`.

- **date (Date)**  
  Fecha específica del calendario.  
  Cada registro corresponde a un día para un alojamiento.

- **available (Boolean)**  
  Indica si el alojamiento está disponible en esa fecha:
  - `true` → disponible para reservar  
  - `false` → no disponible (reservado o bloqueado)

- **minimum_nights (Int32)**  
  Número mínimo de noches requeridas para reservar en esa fecha.

- **maximum_nights (Int32)**  
  Número máximo de noches permitidas para una reserva.

**Consideraciones**

- `available = false` no siempre implica reserva (puede ser bloqueo manual del host)
- Requiere agregación para análisis (datos están a nivel diario)
- Puede ser voluminoso (una fila por día por propiedad)

### Interpretación general

Este dataset permite analizar la **disponibilidad y ocupación diaria de los alojamientos**, ya que:

- Cada fila representa un día específico
- Permite construir el calendario completo de cada propiedad

### Análisis de valores nulos

In [ ]:
analizar_valores_nulos(df_calendar, "CALENDAR")

### Análisis de duplicados

In [ ]:
analizar_duplicados(df_calendar, "CALENDAR", col_id='_id')

### Análisis de outliers

In [ ]:
analizar_outliers(df_calendar, "CALENDAR", columnas=["maximum_nights", "minimum_nights"])

### Análisis de valores negativos

## Conclusiones finales del EDA

### Inconsistencias detectadas

### Correlaciones relevantes

### Decisiones y transformaciones definidas por Colección

**Listings**

**Reviews**

**Calendar**